# Casanovo De Novo Peptide Sequencing — Evaluation Notebook

**Project:** Peptide and Protein Sequencing by Multinomial Diffusion Model   
**Dataset:** E. coli Vesicle Proteomics (Ye Lab)

---

## Overview

This notebook implements the full evaluation pipeline for Casanovo de novo peptide sequencing.
It computes four metrics at two granularity levels, as outlined in the project report and the
Wastewater Proteomics & AI slide deck:

| Level         | Precision | Recall | Accuracy | F1 Score |
|--------------|-----------|--------|----------|----------|
| Amino Acid   | ✓         | ✓      | ✓        | ✓        |
| Peptide      | ✓         | ✓      | ✓        | ✓        |

**Casanovo Hyperparameters (Table 1, project report):**

| Parameter | Value |
|-----------|-------|
| Embedding dimension (dim_model) | 512 |
| Attention heads (n_head) | 8 |
| Feed-forward dim | 1024 |
| Transformer layers | 9 |
| Learning rate | 5×10⁻⁴ |
| Label smoothing | 0.01 |
| Warm-up iterations | 100,000 |
| Cosine decay period | 600,000 |

## 0. Setup & Imports

In [ ]:
# Install required packages (run once)
import subprocess, sys

packages = ["casanovo", "openpyxl", "pandas", "numpy", "matplotlib", "seaborn"]
for pkg in packages:
    try:
        __import__(pkg.split("[")[0])
    except ImportError:
        print(f"Installing {pkg} …")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages ready ✓")

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Add project directory to path so evaluate_metrics is importable
PROJECT_DIR = Path(".").resolve()
sys.path.insert(0, str(PROJECT_DIR))

from evaluate_metrics import (
    load_ground_truth,
    load_casanovo_predictions,
    load_casanovo_csv,
    compute_all_metrics,
    print_metrics_table,
    save_results_csv,
    save_per_spectrum_csv,
    merge_predictions_with_ground_truth,
)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
OUTPUT_DIR = PROJECT_DIR / "casanovo_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Project dir : {PROJECT_DIR}")
print(f"Output dir  : {OUTPUT_DIR}")

---
## 1. Run Casanovo on E. coli mzML Files

Casanovo maps MS/MS spectra → amino acid sequences **without a reference database**  
(see slide deck: *"De novo peptide sequencing"*, and section 4.1 of the project report).

> **GPU note:** Casanovo runs on CPU but is significantly faster with a CUDA GPU.  
> On CPU, expect ~30–60 min per mzML file depending on spectrum count.

In [ ]:
# ── Casanovo hyperparameters (Table 1, project report) ──────────────────────
CASANOVO_CONFIG = {
    "dim_model":                     512,
    "n_head":                        8,
    "dim_feedforward":               1024,
    "n_layers":                      9,
    "dropout":                       0.0,
    "learning_rate":                 5e-4,
    "weight_decay":                  1e-5,
    "train_label_smoothing":         0.01,
    "warmup_iters":                  100_000,
    "cosine_schedule_period_iters":  600_000,
    "predict_batch_size":            1024,
    "precursor_mass_tol":            50,      # ppm
    "isotope_error_range":           [0, 1],
}

print("Casanovo model configuration:")
for k, v in CASANOVO_CONFIG.items():
    print(f"  {k:<40} {v}")

In [ ]:
# ── Download pretrained weights (v5.1.2) ────────────────────────────────────
import urllib.request

MODEL_URL    = "https://github.com/Noble-Lab/casanovo/releases/download/v5.1.2/casanovo_pretrained_v5_1_2.ckpt"
MODEL_PATH   = PROJECT_DIR / "casanovo_pretrained_v5_1_2.ckpt"

if not MODEL_PATH.exists():
    print(f"Downloading model weights → {MODEL_PATH.name} …")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Download complete ✓")
else:
    print(f"Model weights already present: {MODEL_PATH.name} ✓")

In [ ]:
# ── Run Casanovo on EV1 and EV2 ─────────────────────────────────────────────
# Set SKIP_CASANOVO = True if predictions already exist
SKIP_CASANOVO = False

SAMPLES = {
    "EV1": {
        "mzml":  PROJECT_DIR / "Ecoli_EV_1.mzML",
        "xlsx":  PROJECT_DIR / "Database search output_Ecoli_EV_1.xlsx",
        "mztab": OUTPUT_DIR  / "casanovo_EV1_predictions.mztab",
    },
    "EV2": {
        "mzml":  PROJECT_DIR / "Ecoli_EV_2.mzML",
        "xlsx":  PROJECT_DIR / "Database search output_Ecoli_EV_2.xlsx",
        "mztab": OUTPUT_DIR  / "casanovo_EV2_predictions.mztab",
    },
}

CONFIG_PATH = PROJECT_DIR / "casanovo_config.yaml"

if not SKIP_CASANOVO:
    for name, cfg in SAMPLES.items():
        if not cfg["mzml"].exists():
            print(f"[!] {cfg['mzml'].name} not found — skipping {name}")
            continue
        cmd = [
            "casanovo", "sequence",
            "--model",  str(MODEL_PATH),
            "--output", str(cfg["mztab"]),
        ]
        if CONFIG_PATH.exists():
            cmd += ["--config", str(CONFIG_PATH)]
        cmd.append(str(cfg["mzml"]))

        print(f"\nRunning Casanovo on {name} …")
        print("Command:", " ".join(cmd))
        result = subprocess.run(cmd, capture_output=False, text=True)
        if result.returncode == 0:
            print(f"[✓] {name} done → {cfg['mztab'].name}")
        else:
            print(f"[✗] Casanovo failed for {name} (exit code {result.returncode})")
else:
    print("SKIP_CASANOVO=True — using existing prediction files.")
    for name, cfg in SAMPLES.items():
        status = "✓" if cfg["mztab"].exists() else "✗ NOT FOUND"
        print(f"  {name}: {cfg['mztab'].name} [{status}]")

---
## 2. Load Ground Truth (Database Search Results)

The .xlsx files from MaxQuant contain the database-search results for E. coli vesicle
proteomics — these serve as the **ground truth reference** for evaluating Casanovo.

Key columns: `Scan number`, `Sequence`, `Score`, `PEP`

In [ ]:
ground_truths = {}
for name, cfg in SAMPLES.items():
    gt = load_ground_truth(str(cfg["xlsx"]))
    ground_truths[name] = gt
    print(f"\n{name} ground truth: {len(gt)} spectra, {gt['sequence'].nunique()} unique peptides")
    display(gt[["scan_number", "sequence", "score", "pep"]].head(5))

In [ ]:
# Sequence length distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, gt) in zip(axes, ground_truths.items()):
    lengths = gt["sequence"].str.len()
    ax.hist(lengths, bins=range(4, 35), color="steelblue", edgecolor="white", alpha=0.85)
    ax.set_title(f"E. coli {name} — Ground Truth Peptide Length Distribution", fontsize=11)
    ax.set_xlabel("Peptide Length (AA)")
    ax.set_ylabel("Count")
    ax.axvline(lengths.mean(), color="firebrick", linestyle="--", label=f"Mean = {lengths.mean():.1f}")
    ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "gt_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: gt_length_distribution.png")

---
## 3. Load Casanovo Predictions

In [ ]:
predictions = {}
for name, cfg in SAMPLES.items():
    mztab = cfg["mztab"]
    if mztab.exists():
        try:
            preds = load_casanovo_predictions(str(mztab))
            predictions[name] = preds
            print(f"{name}: {len(preds)} predictions loaded from {mztab.name}")
            display(preds[["scan_number", "predicted_sequence", "confidence"]].head(5))
        except Exception as e:
            print(f"[!] Could not parse {mztab.name}: {e}")
    else:
        print(f"[!] {name}: prediction file not found → {mztab}")
        print("    Run Casanovo first (cell above), or set SKIP_CASANOVO=False.")

In [ ]:
# ── Confidence score distribution ────────────────────────────────────────────
if predictions:
    fig, axes = plt.subplots(1, len(predictions), figsize=(6 * len(predictions), 4))
    if len(predictions) == 1:
        axes = [axes]
    for ax, (name, preds) in zip(axes, predictions.items()):
        ax.hist(preds["confidence"], bins=50, color="darkorange", edgecolor="white", alpha=0.85)
        ax.axvline(0.9, color="firebrick", linestyle="--", label="threshold = 0.9")
        ax.set_title(f"Casanovo Confidence Scores — {name}")
        ax.set_xlabel("Confidence")
        ax.set_ylabel("Count")
        ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "confidence_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No predictions loaded yet — run Casanovo first.")

---
## 4. Compute All Metrics

### Metric Definitions

**Amino acid (AA) level** — character-wise comparison using Longest Common Subsequence (LCS):

$$\text{AA Precision} = \frac{\sum_i \text{LCS}(\hat{s}_i, s_i)}{\sum_i |\hat{s}_i|}$$

$$\text{AA Recall} = \frac{\sum_i \text{LCS}(\hat{s}_i, s_i)}{\sum_i |s_i|}$$

$$\text{AA Accuracy} = \frac{\sum_i \text{LCS}(\hat{s}_i, s_i)}{\sum_i \max(|\hat{s}_i|, |s_i|)}$$

**Peptide level** — exact full-sequence match:

$$\text{Peptide Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}} \qquad \text{Peptide Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$

$$\text{Peptide F1} = \frac{2 \cdot P \cdot R}{P + R} \qquad \text{Peptide Accuracy} = \frac{\text{TP}}{\text{TP} + \text{FP} + \text{FN}}$$

In [ ]:
# ── Run evaluation ────────────────────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 0.0   # Change to e.g. 0.9 to keep only high-confidence predictions

all_results = {}

for name in SAMPLES:
    if name not in predictions or name not in ground_truths:
        print(f"Skipping {name} — missing predictions or ground truth.")
        continue

    results = compute_all_metrics(
        predictions[name],
        ground_truths[name],
        confidence_threshold=CONFIDENCE_THRESHOLD,
    )
    all_results[name] = results
    print_metrics_table(results, sample_name=name)

    # Save CSVs
    save_results_csv(results, str(OUTPUT_DIR / f"metrics_{name}.csv"), sample_name=name)
    save_per_spectrum_csv(results, str(OUTPUT_DIR / f"per_spectrum_{name}.csv"))

---
## 5. Metric Visualisations

In [ ]:
# ── 5a. Bar chart — all 8 metrics side by side ───────────────────────────────
if not all_results:
    print("No results yet.")
else:
    metric_labels = {
        "aa_precision":      "AA\nPrecision",
        "aa_recall":         "AA\nRecall",
        "aa_accuracy":       "AA\nAccuracy",
        "aa_f1":             "AA\nF1",
        "peptide_precision": "Pep\nPrecision",
        "peptide_recall":    "Pep\nRecall",
        "peptide_accuracy":  "Pep\nAccuracy",
        "peptide_f1":        "Pep\nF1",
    }

    fig, ax = plt.subplots(figsize=(14, 5))
    x = np.arange(len(metric_labels))
    width = 0.35
    colors = ["#2196F3", "#FF9800"]  # Blue for EV1, Orange for EV2

    for i, (name, results) in enumerate(all_results.items()):
        vals = [results.get(k, 0.0) for k in metric_labels]
        bars = ax.bar(x + i * width - width / 2 * (len(all_results) - 1),
                      vals, width, label=name, color=colors[i % len(colors)],
                      edgecolor="white", alpha=0.88)
        # Value labels on bars
        for bar, v in zip(bars, vals):
            if v > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                        f"{v:.3f}", ha="center", va="bottom", fontsize=7.5, fontweight="bold")

    # Divider between AA and Peptide metrics
    ax.axvline(3.5, color="gray", linestyle="--", alpha=0.5, linewidth=1)
    ax.text(1.5, 1.03, "Amino Acid Level", ha="center", transform=ax.get_xaxis_transform(),
            fontsize=10, color="steelblue", fontweight="bold")
    ax.text(5.5, 1.03, "Peptide Level", ha="center", transform=ax.get_xaxis_transform(),
            fontsize=10, color="darkorange", fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(list(metric_labels.values()), fontsize=9)
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.12)
    ax.set_title("Casanovo De Novo Sequencing — Evaluation Metrics\nE. coli Vesicle Proteomics (Ye Lab)",
                 fontsize=12, fontweight="bold")
    ax.legend(title="Sample")
    ax.yaxis.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "metrics_bar_chart.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: metrics_bar_chart.png")

In [ ]:
# ── 5b. Radar / spider chart ─────────────────────────────────────────────────
if all_results:
    from matplotlib.patches import FancyArrowPatch
    import matplotlib.gridspec as gridspec

    metric_keys = [
        "aa_precision", "aa_recall", "aa_accuracy", "aa_f1",
        "peptide_precision", "peptide_recall", "peptide_accuracy", "peptide_f1",
    ]
    metric_short = ["AA Prec", "AA Rec", "AA Acc", "AA F1",
                    "Pep Prec", "Pep Rec", "Pep Acc", "Pep F1"]

    N = len(metric_keys)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]  # close the polygon

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    colors = ["#2196F3", "#FF9800"]

    for i, (name, results) in enumerate(all_results.items()):
        vals = [results.get(k, 0.0) for k in metric_keys]
        vals += vals[:1]
        ax.plot(angles, vals, color=colors[i], linewidth=2, label=name)
        ax.fill(angles, vals, color=colors[i], alpha=0.15)

    ax.set_thetagrids(np.degrees(angles[:-1]), metric_short, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=8)
    ax.set_title("Casanovo Metrics — Radar Chart\nE. coli EV Proteomics",
                 fontsize=12, fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.15))
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "metrics_radar.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: metrics_radar.png")

In [ ]:
# ── 5c. Per-spectrum AA precision/recall scatter ─────────────────────────────
for name, results in all_results.items():
    per_spec = results.get("_breakdown", {}).get("aa", {}).get("_per_spectrum", [])
    if not per_spec:
        continue
    df_ps = pd.DataFrame(per_spec)

    fig, ax = plt.subplots(figsize=(7, 6))
    sc = ax.scatter(
        df_ps["aa_precision"], df_ps["aa_recall"],
        alpha=0.4, s=20, c=df_ps["lcs"] / df_ps[["pred_len", "gt_len"]].max(axis=1),
        cmap="RdYlGn", vmin=0, vmax=1,
    )
    plt.colorbar(sc, ax=ax, label="AA Accuracy (LCS / max_len)")
    ax.set_xlabel("AA Precision (per spectrum)")
    ax.set_ylabel("AA Recall (per spectrum)")
    ax.set_title(f"Per-Spectrum AA Precision vs Recall — {name}")
    ax.axline((0, 0), slope=1, color="gray", linestyle="--", alpha=0.5, label="P = R")
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"per_spectrum_scatter_{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: per_spectrum_scatter_{name}.png")

In [ ]:
# ── 5d. Peptide-level confusion summary ─────────────────────────────────────
for name, results in all_results.items():
    pep_bd = results.get("_breakdown", {}).get("peptide", {})
    if not pep_bd:
        continue

    tp = pep_bd.get("_tp", 0)
    fp = pep_bd.get("_fp", 0)
    fn = pep_bd.get("_fn", 0)
    total = tp + fp + fn

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Stacked bar
    ax1.bar(["Predictions"], [tp],  color="#4CAF50", label=f"TP = {tp}")
    ax1.bar(["Predictions"], [fp], bottom=[tp], color="#F44336", label=f"FP = {fp}")
    ax1.bar(["Ground Truth"], [tp],  color="#4CAF50")
    ax1.bar(["Ground Truth"], [fn], bottom=[tp], color="#FF9800", label=f"FN = {fn}")
    ax1.set_title(f"Peptide-Level TP/FP/FN — {name}")
    ax1.set_ylabel("Number of Peptides")
    ax1.legend()

    # Metric bar
    m_names = ["Precision", "Recall", "Accuracy", "F1"]
    m_vals  = [
        results["peptide_precision"],
        results["peptide_recall"],
        results["peptide_accuracy"],
        results["peptide_f1"],
    ]
    bar_colors = ["#2196F3", "#009688", "#9C27B0", "#FF5722"]
    bars = ax2.bar(m_names, m_vals, color=bar_colors, edgecolor="white", alpha=0.85)
    for bar, v in zip(bars, m_vals):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{v:.4f}", ha="center", va="bottom", fontweight="bold")
    ax2.set_title(f"Peptide-Level Metrics — {name}")
    ax2.set_ylabel("Score")
    ax2.set_ylim(0, 1.15)

    plt.suptitle(f"E. coli {name} — Casanovo Peptide-Level Evaluation",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"peptide_confusion_{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: peptide_confusion_{name}.png")

---
## 6. Effect of Confidence Threshold on Metrics

Casanovo reports a per-prediction confidence score (0–1). Applying a threshold filters
low-confidence predictions, trading recall for precision — relevant for FDR control.

In [ ]:
# Sweep confidence thresholds and plot metric curves
thresholds = np.arange(0.0, 1.01, 0.05)

for name in SAMPLES:
    if name not in predictions or name not in ground_truths:
        continue

    curve = {k: [] for k in [
        "aa_precision", "aa_recall", "aa_f1",
        "peptide_precision", "peptide_recall", "peptide_f1",
        "n_predictions",
    ]}

    for thr in thresholds:
        preds_thr = predictions[name][predictions[name]["confidence"] >= thr]
        if len(preds_thr) == 0:
            for k in curve:
                curve[k].append(np.nan)
            continue
        r = compute_all_metrics(preds_thr, ground_truths[name])
        for k in ["aa_precision", "aa_recall", "aa_f1",
                  "peptide_precision", "peptide_recall", "peptide_f1"]:
            curve[k].append(r[k])
        curve["n_predictions"].append(len(preds_thr))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(thresholds, curve["aa_precision"], label="AA Precision", color="#2196F3")
    axes[0].plot(thresholds, curve["aa_recall"],    label="AA Recall",    color="#4CAF50")
    axes[0].plot(thresholds, curve["aa_f1"],        label="AA F1",        color="#9C27B0", linestyle="--")
    axes[0].set_title(f"AA-Level Metrics vs Threshold — {name}")
    axes[0].set_xlabel("Confidence Threshold")
    axes[0].set_ylabel("Score")
    axes[0].legend()
    axes[0].set_ylim(0, 1.05)

    axes[1].plot(thresholds, curve["peptide_precision"], label="Pep Precision", color="#FF5722")
    axes[1].plot(thresholds, curve["peptide_recall"],    label="Pep Recall",    color="#009688")
    axes[1].plot(thresholds, curve["peptide_f1"],        label="Pep F1",        color="#795548", linestyle="--")
    axes[1].set_title(f"Peptide-Level Metrics vs Threshold — {name}")
    axes[1].set_xlabel("Confidence Threshold")
    axes[1].set_ylabel("Score")
    axes[1].legend()
    axes[1].set_ylim(0, 1.05)

    axes[2].plot(thresholds, curve["n_predictions"], color="#607D8B")
    axes[2].set_title(f"# Predictions Retained vs Threshold — {name}")
    axes[2].set_xlabel("Confidence Threshold")
    axes[2].set_ylabel("# Predictions")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"threshold_sweep_{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: threshold_sweep_{name}.png")

---
## 7. Combined Summary Table

In [ ]:
metric_keys = [
    "aa_precision", "aa_recall", "aa_accuracy", "aa_f1",
    "peptide_precision", "peptide_recall", "peptide_accuracy", "peptide_f1",
]

rows = []
for name, results in all_results.items():
    row = {"Sample": name}
    row.update({k: round(results.get(k, float("nan")), 4) for k in metric_keys})
    rows.append(row)

if rows:
    summary_df = pd.DataFrame(rows).set_index("Sample")
    summary_df.columns = [
        "AA Precision", "AA Recall", "AA Accuracy", "AA F1",
        "Pep Precision", "Pep Recall", "Pep Accuracy", "Pep F1",
    ]

    # Style the table
    styled = summary_df.style \
        .background_gradient(cmap="RdYlGn", vmin=0, vmax=1) \
        .format("{:.4f}") \
        .set_caption("Casanovo — All Metrics Summary (E. coli EV Proteomics)")
    display(styled)

    # Save combined CSV
    summary_df.to_csv(OUTPUT_DIR / "metrics_combined.csv")
    print(f"\nSaved: {OUTPUT_DIR / 'metrics_combined.csv'}")
else:
    print("No results to display — run evaluation cells above.")

---
## 8. Interpretation Notes

**Context from the project (PDF + slide deck):**

- The E. coli data workflow: database search results (MaxQuant) → ground truth; Casanovo mzML predictions → evaluated here.
- **AA-level recall** measures how many correct amino acids Casanovo recovered. Even partially correct predictions contribute — important for downstream protein identification via UniPept.
- **Peptide-level precision** is stricter: Casanovo must get the *exact* sequence right. Expect this to be lower than AA-level metrics.
- **Confidence threshold trade-off**: Higher threshold → higher peptide precision, lower recall. The paper recommends ~0.9 for FDR control at 1%.
- **Isoleucine / Leucine (I/L)**: Mass spectrometry cannot distinguish I from L. Both map to the same m/z. Consider treating I=L during evaluation for fairness.
- **Next steps**: Compare with Novor, InstaNovo, InstaNovo+ on the same spectra; visualise in UniPept for taxonomic interpretation.

**Key Casanovo hyperparameter insights (Table 1):**
- Large model (dim=512, 9 layers) trained on 30M spectra — dropout=0 is intentional (data regularises).
- Label smoothing (0.01) calibrates per-residue confidence scores — confidence values are interpretable.
- Cosine LR decay over 600K iters ensures gradual convergence without catastrophic forgetting.

In [ ]:
print("\n=== Files saved to casanovo_outputs/ ===")
for f in sorted(OUTPUT_DIR.glob("*")):
    print(f"  {f.name}")